### Text preprocess

In [ ]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode
import pandas as pd
#stemmer = nltk.RSLPStemmer()


def proccess_text(tweets):
    
    # Removing links, mentions and hashtags
    tweets['processed_text'] = tweets.text.str.replace(r'(http\S+)', '',regex=True) \
                                          .str.replace(r'@[\w]*', '',regex=True) \
                                          .str.replace(r'#[\w]*','',regex=True) 
    print('[ok] - Removing links.')
    print('[ok] - Removing mentions.')
    print('[ok] - Removing hashtags.')

    textWords = ' '.join([text for text in tweets.processed_text])

    # Removing accent
    textWords = [unidecode.unidecode(text) for text in tweets.processed_text ]    
    print('[ok] - Removing accent.')
    
    # Creating a list of words and characters (stopwords) to be removed from the text
    # stopWords = nltk.corpus.stopwords.words("portuguese")    
    print('[ok] - Creating a list of words and characters (stopwords) to be removed from the text.')
    
    
    # Separating punctuation from words
    punctSeparator = tokenize.WordPunctTokenizer()
    punctuationList = list()
    for punct in punctuation:
        punctuationList.append(punct)
        
    #stopWords =   punctuationList + stopWords    
    stopWords =   punctuationList
    #print('[ok] - Separating punctuation from words.')


    # Iterating over the text and removing stop words 
    trasnformedText = list()    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = punctSeparator.tokenize(text)
        for words in textWords:
             if words not in stopWords:
                #newText.append(stemmer.stem(words))
                newText.append(words)
        trasnformedText.append(' '.join(newText))
    tweets.processed_text = trasnformedText
    print('[ok] - Removing punctuation and set text to lowecase.')
   
    # Removing all non-text characters
    tweets.processed_text = tweets['processed_text'].str.replace(r"[^a-zA-Z#]", " ", regex=True)                                                         
    print('[ok] - Removing all non-text characters.')
   
    trasnformedText = list()
    for phrase in tweets.processed_text:
        newPhrase = list()   
        newPhrase.append(' '.join(phrase.split()))
        for words in newPhrase:
            trasnformedText.append(''.join(newPhrase))
    tweets.processed_text = trasnformedText
    
    # Removing tweets with less than three terms
    index=[x for x in tweets.index if tweets.processed_text[x].count(' ') < 3]
    tweets = tweets.drop(index)
    print('[ok] - Removing tweets with less than three terms.')

    # Removing empty lines
    removeEmpty  = tweets.processed_text != ' '
    tweets = tweets[removeEmpty]
    print('[ok] - Removing empty lines.')

    tweets.reset_index(inplace=True)
    #tweets = {'created_at': tweets.created_at, 'id':tweets.id,'author_id':tweets.author_id,'in_reply_to_user_id':tweets.in_reply_to_user_id, 'text': tweets.processed_text}
    tweets = {'text': tweets.processed_text}
    tweets = pd.DataFrame(tweets)
    #tweets = tweets.sort_values(['created_at']).reset_index().drop(columns=["index"])
    #tweets = tweets.reset_index().drop(columns=["index"])
    
    return tweets

In [ ]:
import pandas as pd 

trump2016 = pd.read_csv('../data/raw/trump2016.csv')
trump2016 = trump2016[['text']]
trump2016 = trump2016.dropna()



donaldtrump = pd.read_csv(open('../data/raw/donaldtrump.csv','rU'), encoding='utf-8')
donaldtrump = donaldtrump[['text']]
donaldtrump = donaldtrump.dropna()


docs = pd.concat([trump2016,donaldtrump])
docs = docs.reset_index().drop(columns=['index'])

In [ ]:
tweets = proccess_text(docs)

In [ ]:
tweets.to_csv('../data/donaldtrump.csv', index=False)

### Extract more frequent hashtags

In [ ]:
import re
import nltk
from nltk import tokenize

def count_hashtags(data_of_text):
    regex = re.compile(r'#[\w]*')
    #regex = re.compile(r'#(\w*[0-9a-zA-Z]+\w*[0-9a-zA-Z])')

    textWords = ' '.join([text for text in data_of_text])

    hashtags = regex.findall(textWords)

    hashtags = ' '.join([text for text in  hashtags])

    tokenizing = tokenize.WhitespaceTokenizer()
    tokenizedWords = tokenizing.tokenize(hashtags)
    frequency = nltk.FreqDist(tokenizedWords)
    df_frequency = pd.DataFrame({"Hashtag": list(frequency.keys()),
                                       "Frequency": list(frequency.values())})

    return df_frequency

In [ ]:
import nltk
from nltk import tokenize
import numpy as np 
from string import punctuation
import unidecode


    #tokens = tokenize.WordPunctTokenizer()
    #tokens = tokenize.WhitespaceTokenizer()
    #tokens = tokenize.MWETokenizer()

def proccess_text(data_of_text, tk):
    textWords = [unidecode.unidecode(text) for text in data_of_text]       
    
    
    punctuationList = list()
    for punct in punctuation:
        if punct != '#':
            punctuationList.append(punct)
    trasnformedText = list()
    
    personalList=["...","!#","@#","'#",".#","\"#","...#","(#","?#","!!"]  
    punctuationList = punctuationList  
    
    for text in textWords:
        newText = list()   
        text = text.lower()
        textWords = tk.tokenize(text)
        for words in textWords: 
             if words not in punctuationList:
                 newText.append(words)
        trasnformedText.append(' '.join(newText))
    return trasnformedText

In [ ]:
import pandas as pd

tweets = pd.read_csv('../data/raw/trump_raw.csv')
tweets

In [ ]:
tweets['processed_text'] = proccess_text(tweets.text, tokenize.WhitespaceTokenizer())
tweets


In [ ]:
df = count_hashtags(tweets.processed_text)
df

In [ ]:
import tabloo

tabloo.show(df)